In [ ]:
!pwd


In [ ]:
!nvidia-smi


# Create Environment


In [ ]:
!test -d /content/edge-lipsync-model || git clone https://github.com/monkira99/edge-lipsync-model.git /content/edge-lipsync-model


In [ ]:
%cd /content/edge-lipsync-model


In [ ]:
!git pull


In [ ]:
!uv sync


In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or environment before running Hub commands."


In [ ]:
!hf auth login --token "$HF_TOKEN" --quiet

!uv run python tools/hf_model_assets.py pull \
  --repo-id tiennguyenbnbk/edge-lipsync-model-assets \
  --local-dir models


In [ ]:
!ls -lah /content/edge-lipsync-model/models/mediapipe


# Build Silent/Talking Pose-Paired Dataset Snapshot

Build the V1 dataset from `data/<persona>/silent/defaultvideo.mp4` and `data/<persona>/talking/*.mp4`.

The output snapshot is self-contained: `source_roi`, `target_roi`, and `audio [20,256]` are embedded so another machine can train with `snapshot_download()` + `load_from_disk()` without the source MP4 files.


In [ ]:
from pathlib import Path
import json
import yaml

ROOT = Path("/content/edge-lipsync-model")
PERSONA_ID = "nora"

# Preferred Colab layout. If this repo already contains data/<persona>, use it as a fallback.
DATA_ROOT = Path("/content/data")
if not (DATA_ROOT / PERSONA_ID).exists() and (ROOT / "data" / PERSONA_ID).exists():
    DATA_ROOT = ROOT / "data"

SNAPSHOT_ROOT = Path(f"/content/data/datasets/{PERSONA_ID}_pose_pairs")
WORK_ROOT = Path(f"/content/data/work/{PERSONA_ID}_pose_pairs")
HF_CACHE_DIR = Path("/content/data/hf_cache")
OUTPUT_REPO = f"tiennguyenbnbk/{PERSONA_ID}-silent-talking-pose-pairs"

WENET_ONNX = ROOT / "models/wenet/wenet.onnx"
FACE_LANDMARKER = ROOT / "models/mediapipe/face_landmarker.task"
BUILD_CONFIG_PATH = ROOT / "configs/silent_talking_dataset.colab.yaml"
PUSH_LOG = Path("/content/data/silent_talking_dataset_push.log")

for directory in (SNAPSHOT_ROOT.parent, WORK_ROOT.parent, HF_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"data_root={DATA_ROOT}")
print(f"persona_id={PERSONA_ID}")
print(f"snapshot_root={SNAPSHOT_ROOT}")
print(f"work_root={WORK_ROOT}")
print(f"output_repo={OUTPUT_REPO}")


In [ ]:
silent_video = DATA_ROOT / PERSONA_ID / "silent/defaultvideo.mp4"
talking_dir = DATA_ROOT / PERSONA_ID / "talking"
talking_videos = sorted(talking_dir.glob("*.mp4")) if talking_dir.exists() else []

assert silent_video.is_file(), f"Missing silent video: {silent_video}"
assert talking_videos, f"Missing talking videos under: {talking_dir}"
assert WENET_ONNX.is_file(), f"Missing Wenet ONNX: {WENET_ONNX}"
assert FACE_LANDMARKER.is_file(), f"Missing MediaPipe face landmarker: {FACE_LANDMARKER}"

print(f"silent={silent_video}")
print(f"talking_videos={len(talking_videos)}")
for path in talking_videos[:10]:
    print(f"- {path.name}")


In [ ]:
build_config = {
    "data_root": str(DATA_ROOT),
    "persona_id": PERSONA_ID,
    "snapshot_root": str(SNAPSHOT_ROOT),
    "work_root": str(WORK_ROOT),
    "wenet_onnx": str(WENET_ONNX),
    "landmark_model_asset_path": str(FACE_LANDMARKER),
    "fps": 25,
    "sample_rate": 16000,
    "validation_fraction": 0.20,
    "split_salt": "edge-lipsync-silent-talking-v1",
    "idle_max_ratio": 0.10,
    "idle_sample_weight": 0.25,
    "preview_count_per_group": 8,
    "progress": False,
    "strict": True,
    "matching": {
        "max_yaw_delta": 5.0,
        "max_pitch_delta": 5.0,
        "max_roll_delta": 4.0,
        "max_center_x_delta": 0.05,
        "max_center_y_delta": 0.05,
        "min_width_ratio": 0.9,
        "max_width_ratio": 1.1,
        "min_height_ratio": 0.9,
        "max_height_ratio": 1.1,
        "pose_weight": 1.0,
        "position_weight": 1.0,
        "scale_weight": 1.0,
    },
    "post_crop_alignment": {
        "max_stable_landmark_rmse": 0.04,
        "max_mouth_center_delta": 0.04,
    },
    "blur": {
        "min_source_face_laplacian_variance": 60.0,
        "min_target_face_laplacian_variance": 60.0,
        "min_target_mouth_laplacian_variance": 40.0,
    },
    "sync": {
        "window_seconds": 2.0,
        "stride_seconds": 1.0,
        "max_lag_frames": 3,
        "max_reject_lag_frames": 2,
        "min_correlation": 0.20,
        "silence_rms_threshold": 0.001,
        "speech_fraction_threshold": 0.25,
    },
}

BUILD_CONFIG_PATH.write_text(yaml.safe_dump(build_config, sort_keys=False), encoding="utf-8")
print(BUILD_CONFIG_PATH)


In [ ]:
!sed -n "1,240p" {BUILD_CONFIG_PATH}


In [ ]:
!uv run python tools/build_silent_talking_dataset.py \
  --config {BUILD_CONFIG_PATH} \
  --strict \
  --no-progress


In [ ]:
from datasets import Image, load_from_disk

metadata = json.loads((SNAPSHOT_ROOT / "build_complete.json").read_text())
dataset = load_from_disk(SNAPSHOT_ROOT / "dataset")
raw_roi = dataset["train"].cast_column("source_roi", Image(decode=False))[0]["source_roi"]

print(json.dumps({
    "schema_version": metadata["schema_version"],
    "train_rows": metadata["train_rows"],
    "val_rows": metadata["val_rows"],
    "talking_clips": metadata["talking_clips"],
    "failed_clips": metadata["failed_clips"],
    "split_mode": metadata["split_mode"],
    "dataset_fingerprints": metadata["dataset_fingerprints"],
}, indent=2))
print(dataset)
print(f"source_roi_path={raw_roi['path']}")
print(f"source_roi_png_magic={raw_roi['bytes'][:8]}")


In [ ]:
!uv run python tools/hf_dataset.py push-snapshot \
  --snapshot-root {SNAPSHOT_ROOT} \
  --repo-id {OUTPUT_REPO} | tee {PUSH_LOG}


In [ ]:
push_lines = PUSH_LOG.read_text(encoding="utf-8").splitlines()
revision_line = next(line for line in push_lines if line.startswith("revision="))
DATASET_REVISION = revision_line.split("=", 1)[1].strip()
LOCAL_SNAPSHOT_ROOT = Path("/content/data/hf_snapshots") / OUTPUT_REPO.split("/")[-1] / DATASET_REVISION

print(f"HF_DATASET_REPO = {OUTPUT_REPO!r}")
print(f"HF_DATASET_REVISION = {DATASET_REVISION!r}")
print(f"HF_DATASET_LOCAL_DIR = {str(LOCAL_SNAPSHOT_ROOT)!r}")


In [ ]:
!uv run python tools/hf_dataset.py pull-snapshot \
  --repo-id {OUTPUT_REPO} \
  --revision {DATASET_REVISION} \
  --local-dir {LOCAL_SNAPSHOT_ROOT} \
  --cache-dir {HF_CACHE_DIR}


In [ ]:
print("Copy these values into notebooks/training_dev.ipynb:")
print(f"HF_DATASET_REPO = {OUTPUT_REPO!r}")
print(f"HF_DATASET_REVISION = {DATASET_REVISION!r}")
print(f"HF_DATASET_LOCAL_DIR = {str(LOCAL_SNAPSHOT_ROOT)!r}")
